# Philadelphia Data Cleaning and Loading

This notebook loads the Yelp raw files, keeps Philadelphia restaurants only, creates review/check-in/photo features, and saves the processed datasets used by the regression notebook.


In [ ]:
import math
import sys
from pathlib import Path

import pandas as pd
from textblob import TextBlob

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

from add_engagement_features import build_checkin_features, build_photo_features

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

target_city = "Philadelphia"

business_path = raw_dir / "yelp_academic_dataset_business.json"
review_path = raw_dir / "yelp_academic_dataset_review.json"

restaurants_clean_path = processed_dir / "restaurants_cleaned_philadelphia.csv"
restaurant_review_features_path = processed_dir / "restaurant_review_features_philadelphia.csv"
enriched_features_path = processed_dir / "restaurant_review_checkin_photo_features_philadelphia.csv"


In [ ]:
# Load businesses and keep Philadelphia restaurants only.
business = pd.read_json(business_path, lines=True)

restaurants = business[
    business["categories"].fillna("").str.contains("Restaurants", case=False, na=False)
].copy()
restaurants = restaurants[
    restaurants["city"].fillna("").str.casefold() == target_city.casefold()
].copy()


def attribute_is_true(attributes, key):
    if not isinstance(attributes, dict):
        return 0
    return int(str(attributes.get(key, "")).strip().lower() == "true")


restaurants["review_count"] = pd.to_numeric(restaurants["review_count"], errors="coerce").fillna(0)
restaurants["log_review_count"] = restaurants["review_count"].apply(math.log1p)
restaurants["has_takeout"] = restaurants["attributes"].apply(lambda attrs: attribute_is_true(attrs, "RestaurantsTakeOut"))
restaurants["has_delivery"] = restaurants["attributes"].apply(lambda attrs: attribute_is_true(attrs, "RestaurantsDelivery"))
restaurants["number_of_categories"] = restaurants["categories"].fillna("").apply(
    lambda value: len([category.strip() for category in value.split(",") if category.strip()])
)

restaurants.to_csv(restaurants_clean_path, index=False)
print(f"Saved {len(restaurants):,} Philadelphia restaurants to {restaurants_clean_path}")
restaurants.head()


In [ ]:
def count_words(text):
    return len(str(text).split())


def get_sentiment(text):
    return TextBlob(str(text)).sentiment.polarity


# Create business-level review features in chunks because the review file is large.
restaurant_ids = set(restaurants["business_id"].dropna())
chunk_size = 50_000
chunk_summaries = []

for chunk_number, review_chunk in enumerate(pd.read_json(review_path, lines=True, chunksize=chunk_size), start=1):
    restaurant_reviews = review_chunk[review_chunk["business_id"].isin(restaurant_ids)].copy()
    if restaurant_reviews.empty:
        continue

    for column in ["stars", "useful", "funny", "cool"]:
        restaurant_reviews[column] = pd.to_numeric(restaurant_reviews[column], errors="coerce").fillna(0)

    restaurant_reviews["review_word_count"] = restaurant_reviews["text"].apply(count_words)
    restaurant_reviews["sentiment"] = restaurant_reviews["text"].apply(get_sentiment)

    chunk_summary = (
        restaurant_reviews
        .groupby("business_id", as_index=False)
        .agg(
            review_stars_sum=("stars", "sum"),
            review_word_count_sum=("review_word_count", "sum"),
            useful_sum=("useful", "sum"),
            funny_sum=("funny", "sum"),
            cool_sum=("cool", "sum"),
            sentiment_sum=("sentiment", "sum"),
            total_review_count=("review_id", "count"),
        )
    )
    chunk_summaries.append(chunk_summary)

    if chunk_number % 25 == 0:
        print(f"Processed {chunk_number:,} review chunks")

if chunk_summaries:
    review_sums = pd.concat(chunk_summaries, ignore_index=True)
    business_review_sums = review_sums.groupby("business_id", as_index=False).sum()

    business_review_sums["avg_review_stars"] = business_review_sums["review_stars_sum"] / business_review_sums["total_review_count"]
    business_review_sums["avg_review_length"] = business_review_sums["review_word_count_sum"] / business_review_sums["total_review_count"]
    business_review_sums["avg_useful"] = business_review_sums["useful_sum"] / business_review_sums["total_review_count"]
    business_review_sums["avg_funny"] = business_review_sums["funny_sum"] / business_review_sums["total_review_count"]
    business_review_sums["avg_cool"] = business_review_sums["cool_sum"] / business_review_sums["total_review_count"]
    business_review_sums["avg_sentiment"] = business_review_sums["sentiment_sum"] / business_review_sums["total_review_count"]

    review_features = business_review_sums[
        [
            "business_id",
            "avg_review_stars",
            "avg_review_length",
            "avg_useful",
            "avg_funny",
            "avg_cool",
            "total_review_count",
            "avg_sentiment",
        ]
    ]
else:
    review_features = pd.DataFrame(
        columns=[
            "business_id",
            "avg_review_stars",
            "avg_review_length",
            "avg_useful",
            "avg_funny",
            "avg_cool",
            "total_review_count",
            "avg_sentiment",
        ]
    )

restaurants_with_review_features = restaurants.merge(review_features, on="business_id", how="left")
restaurants_with_review_features.to_csv(restaurant_review_features_path, index=False)
print(f"Saved review features to {restaurant_review_features_path}")
restaurants_with_review_features.head()


In [ ]:
# Add check-in and photo features to the Philadelphia restaurant-level dataset.
restaurant_ids = set(restaurants_with_review_features["business_id"].dropna())

checkin_features = build_checkin_features(raw_dir, restaurant_ids)
photo_features = build_photo_features(raw_dir, restaurant_ids)

enriched = restaurants_with_review_features.merge(checkin_features, on="business_id", how="left")
enriched = enriched.merge(photo_features, on="business_id", how="left")

fill_zero_columns = [
    "checkin_count",
    "active_checkin_days",
    "photo_count",
    "photo_file_count",
    "photo_file_coverage",
    "avg_image_brightness",
    "avg_image_saturation",
    "food_photo_share",
    "inside_photo_share",
    "outside_photo_share",
    "drink_photo_share",
    "menu_photo_share",
]
enriched[fill_zero_columns] = enriched[fill_zero_columns].fillna(0)

enriched.to_csv(enriched_features_path, index=False)
print(f"Saved {len(enriched):,} enriched Philadelphia rows to {enriched_features_path}")
enriched.head()


In [ ]:
# Final sanity checks.
print(enriched["city"].value_counts(dropna=False))
print(enriched.shape)
enriched.describe(include="all").T.head(20)
